# Adaptive Adversarial Threat Detection and Defense Framework
## Notebook 3 — Adversarial Training Defense (Week 6)

**Prerequisites:** Run `week-4/notebook-baseline.ipynb` (Cells 1–12) and
`week-5/notebook-attack.ipynb` (Cells 1–5) first so that `X_train`, `X_test`,
`y_train`, `y_test`, `rf_baseline`, `top5_idx`, `DEFENSE_NOISE_STD`,
and all C1/C2 metric variables are in memory.

In [ ]:
# ─── Cell 1: Confirm prerequisites are available ──────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
print('Prerequisites check:')
print(f'  X_train shape:      {X_train.shape}')
print(f'  DEFENSE_NOISE_STD:  {DEFENSE_NOISE_STD}')
print(f'  top5_idx:           {top5_idx}')
print(f'  C1 recall:          {rec_c1:.4f}')
print(f'  C2 recall:          {rec_c2:.4f}')

In [ ]:
# ─── Cell 2: Function to create adversarial TRAINING samples ──────────────────
# Note: this is different from create_adversarial_test().
# Here we perturb TRAINING set attack samples (not test set).

def create_adversarial_train(X_train, y_train, top5_idx, noise_std, random_state=42):
    """
    Create adversarial copies of attack-class TRAINING samples.
    Labels are preserved as 1 (attack) — the model learns that
    perturbed attacks are still attacks.
    """
    np.random.seed(random_state)
    attack_mask = (y_train == 1)
    X_attack    = X_train[attack_mask].copy()       # only attack rows
    y_attack    = y_train[attack_mask].copy()       # all labels = 1
    noise = np.random.normal(
        loc=0,
        scale=noise_std,
        size=(X_attack.shape[0], len(top5_idx))
    )
    X_attack[:, top5_idx] += noise                  # perturb top-5 features only
    return X_attack, y_attack

print('create_adversarial_train() defined.')

In [ ]:
# ─── Cell 3: Build the augmented training set ─────────────────────────────────
X_adv_train, y_adv_train = create_adversarial_train(
    X_train, y_train, top5_idx, noise_std=DEFENSE_NOISE_STD
)

# Stack: original training data + adversarial attack copies
X_train_augmented = np.vstack([X_train, X_adv_train])
y_train_augmented = np.concatenate([y_train, y_adv_train])

print(f'Original training set:     {X_train.shape}')
print(f'Adversarial samples added: {X_adv_train.shape}')
print(f'Augmented training set:    {X_train_augmented.shape}')
print(f'Augmented class balance — 0 (normal): {(y_train_augmented==0).sum()} | 1 (attack): {(y_train_augmented==1).sum()}')

In [ ]:
# ─── Cell 4: Train the defended Random Forest ─────────────────────────────────
# Same hyperparameters as baseline — ensures fair comparison.
rf_defended = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_defended.fit(X_train_augmented, y_train_augmented)
print('Defended model training complete.')

In [ ]:
# ─── Cell 5: Evaluate defended model on the SAME adversarial test set as C2 ───
# IMPORTANT: must use same adversarial test set used for C2 — recreate if needed.

def create_adversarial_test(X_test, y_test, top5_idx, noise_std, random_state=42):
    np.random.seed(random_state)
    X_adv = X_test.copy()
    attack_rows = np.where(y_test == 1)[0]
    noise = np.random.normal(0, noise_std, size=(len(attack_rows), len(top5_idx)))
    X_adv[np.ix_(attack_rows, top5_idx)] += noise
    return X_adv

X_adv_test = create_adversarial_test(X_test, y_test, top5_idx, noise_std=DEFENSE_NOISE_STD)

y_pred_defended = rf_defended.predict(X_adv_test)
y_prob_defended = rf_defended.predict_proba(X_adv_test)[:, 1]

acc_c3  = accuracy_score(y_test, y_pred_defended)
prec_c3 = precision_score(y_test, y_pred_defended)
rec_c3  = recall_score(y_test, y_pred_defended)
f1_c3   = f1_score(y_test, y_pred_defended)
auc_c3  = roc_auc_score(y_test, y_prob_defended)

print('=== CONDITION 3: POST-DEFENSE RESULTS ===')
print(f'Accuracy : {acc_c3:.4f}')
print(f'Precision: {prec_c3:.4f}')
print(f'Recall   : {rec_c3:.4f}')
print(f'F1-Score : {f1_c3:.4f}')
print(f'AUC-ROC  : {auc_c3:.4f}')

In [ ]:
# ─── Cell 6: Three-condition summary table ────────────────────────────────────
print('=== FINAL THREE-CONDITION COMPARISON ===')
print(f'{"Condition":>25} | {"Accuracy":>10} | {"Precision":>10} | {"Recall":>8} | {"F1":>8} | {"AUC":>8}')
print('-' * 80)
print(f'{"C1: Baseline":>25} | {acc_c1:>10.4f} | {prec_c1:>10.4f} | {rec_c1:>8.4f} | {f1_c1:>8.4f} | {auc_c1:>8.4f}')
print(f'{"C2: Under Attack":>25} | {acc_c2:>10.4f} | {"N/A":>10} | {rec_c2:>8.4f} | {f1_c2:>8.4f} | {auc_c2:>8.4f}')
print(f'{"C3: Post-Defense":>25} | {acc_c3:>10.4f} | {prec_c3:>10.4f} | {rec_c3:>8.4f} | {f1_c3:>8.4f} | {auc_c3:>8.4f}')
print()
print(f'Recall drop   C1→C2: {rec_c1 - rec_c2:+.4f}')
print(f'Recall recovery C2→C3: {rec_c3 - rec_c2:+.4f}')
print(f'Net change    C1→C3: {rec_c3 - rec_c1:+.4f}')

---
## Visualizations

In [ ]:
# ─── Cell 7: Three-condition grouped bar chart ────────────────────────────────
metrics_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
c1_vals = [acc_c1, prec_c1, rec_c1, f1_c1, auc_c1]
c2_vals = [acc_c2, None,    rec_c2, f1_c2, auc_c2]   # prec_c2 not stored — use N/A
c3_vals = [acc_c3, prec_c3, rec_c3, f1_c3, auc_c3]

# Use 0 where precision for C2 not stored
c2_vals_plot = [v if v is not None else 0 for v in c2_vals]

x     = np.arange(len(metrics_labels))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width, c1_vals,      width, label='C1: Baseline',     color='steelblue')
bars2 = ax.bar(x,         c2_vals_plot, width, label='C2: Under Attack',  color='crimson')
bars3 = ax.bar(x + width, c3_vals,      width, label='C3: Post-Defense',  color='forestgreen')

ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title(f'Three-Condition Performance Comparison\n(Gaussian noise std={DEFENSE_NOISE_STD})', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(metrics_labels)
ax.set_ylim(0, 1.15)
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bar in [bars1, bars2, bars3]:
    for b in bar:
        if b.get_height() > 0:
            ax.annotate(f'{b.get_height():.2f}',
                        xy=(b.get_x() + b.get_width()/2, b.get_height()),
                        xytext=(0, 3), textcoords='offset points',
                        ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('comparison-barchart.png', dpi=150)
plt.show()
print('Saved: comparison-barchart.png')

In [ ]:
# ─── Cell 8: Confusion matrix for Condition 3 ─────────────────────────────────
cm_defense = confusion_matrix(y_test, y_pred_defended)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_defense, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Normal', 'Attack'],
            yticklabels=['Normal', 'Attack'])
plt.title(f'Confusion Matrix — Condition 3: Post-Defense (std={DEFENSE_NOISE_STD})')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion-matrix-defense.png', dpi=150)
plt.show()
print('Saved: confusion-matrix-defense.png')

In [ ]:
# ─── Cell 9: Final results summary — copy to comparison-table.md ──────────────
print('╔══════════════════════════════════════════════════════════════════════╗')
print('║      WEEK 6 FINAL RESULTS — COPY THESE TO comparison-table.md       ║')
print('╠══════════════════════════════════════════════════════════════════════╣')
print(f'║  C1 Baseline:   Acc={acc_c1:.4f} Prec={prec_c1:.4f} Rec={rec_c1:.4f} F1={f1_c1:.4f} AUC={auc_c1:.4f} ║')
print(f'║  C2 Under Atk:  Acc={acc_c2:.4f}            Rec={rec_c2:.4f} F1={f1_c2:.4f} AUC={auc_c2:.4f} ║')
print(f'║  C3 Post-Def:   Acc={acc_c3:.4f} Prec={prec_c3:.4f} Rec={rec_c3:.4f} F1={f1_c3:.4f} AUC={auc_c3:.4f} ║')
print('╠══════════════════════════════════════════════════════════════════════╣')
print(f'║  Recall drop C1→C2:      {rec_c1 - rec_c2:+.4f}                                   ║')
print(f'║  Recall recovery C2→C3:  {rec_c3 - rec_c2:+.4f}                                   ║')
print(f'║  Net change C1→C3:       {rec_c3 - rec_c1:+.4f}                                   ║')
print('╚══════════════════════════════════════════════════════════════════════╝')